In this notebook we explore several ML approaches to classifying Alzheimer's disease (AD) and Parkinson's disease (PD) on the basis of vocal production features using the [Bridge2AI-Voice dataset](https://doi.org/10.13026/8xbn-nq66) 2.0.1 release. 

The high-level goal would be to develop a convenient and accessible assessment utility integrated into the existing proliferation of voice recording tools.

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

'''
foreshadowing the methods we'll be trying: 
logistic regression, gradient boosting, svc, and mlp 
plus some attempts at dimensionality reduction
'''
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA

#we use this to facilitate the mlp model selection
from itertools import combinations

#and this to make a tidier notebook
#import warnings
#warnings.filterwarnings(action="ignore",)

#I used numbergenerator.org to generate this number which will be my seed
rng = 48396

## load in and preprocess data

Described at length in their report, the Bridge2AI-Voice (B2AIv) dataset includes audio features derived from participant vocal production recordings; participant recruitment was initiated upon their presentation at "specialty clinics and institutions" and was based on membership in predetermined groups including "Neurological disorders", to which the AD and PD participants belong. 

Therefore, the data we're working with here are the audio derivative feature data ("static features.tsv") associated with AD, PD, and control participants (a subset of "phenotype.tsv"). 

NB: this work is all in the 2.0.1 data release, which is one release behind at time of writing. 

In [43]:
#load in
pheno = pd.read_csv(r'bridge2ai-2.0.1/phenotype.tsv',sep='\t',header=0)
statics = pd.read_csv(r'bridge2ai-2.0.1/static_features.tsv',sep='\t',header=0)

#subset participants according to AD, PD, or control group
pd_pids = pheno[pheno['diagnosis_parkinsons_gsd_category_1_calculation']=='Yes'].participant_id.unique()
ad_pids = pheno[pheno['alz_dementia_mci']=='Checked'][['diagnosis_alz_dementia_mci_dt','participant_id']].dropna().participant_id.unique()
ct_pids = pheno[pheno['is_control_participant']=='Yes'].participant_id.unique()

In [47]:
#make sure we don't have overlap
sum(np.isin(pd_pids,ad_pids)),sum(np.isin(pd_pids,ct_pids)),sum(np.isin(ad_pids,ct_pids))

(0, 0, 0)

In [37]:
#how many participants do we have in each condition?
print(f'{len(np.unique(pd_pids))} PD participants \n{len(np.unique(ad_pids))} AD participants \n{len(np.unique(ct_pids))} Control participants')

31 PD participants 
37 AD participants 
64 Control participants


In [50]:
#subset static features from participants of interest
pd_stats = statics[statics['participant_id'].isin(pd_pids)]
ad_stats = statics[statics['participant_id'].isin(ad_pids)]
ct_stats = statics[statics['participant_id'].isin(ct_pids)]

'''
participants performed a variety of tasks for the B2AIv data.
to honor our high-level aim, we opt to use the most naturalistic one: free speech
we'll tidy up the data a bit and assign labels here too
'''
pd_fs = pd_stats[pd_stats['task_name'].isin(['Free-Speech','Free-speech-1','Free-speech-2','Free-speech-3'])][[i for i in pd_stats.columns if i not in ['session_id','task_name','transcription']]].dropna()
pd_fs['label'] = 1
ad_fs = ad_stats[ad_stats['task_name'].isin(['Free-Speech','Free-speech-1','Free-speech-2','Free-speech-3'])][[i for i in ad_stats.columns if i not in ['session_id','task_name','transcription']]].dropna()
ad_fs['label'] = 2
ct_fs = ct_stats[ct_stats['task_name'].isin(['Free-Speech','Free-speech-1','Free-speech-2','Free-speech-3'])][[i for i in ct_stats.columns if i not in ['session_id','task_name','transcription']]].dropna()
ct_fs['label'] = 0

In [57]:
'''
some participants contribute multiple data to the set, so we split
by participant to prevent leakage. We went with a 70/30 split 
to keep at least 10 diagnosis-bearing participants 
in the validation set for each diagnosis condition. 
'''

train_parts_pd = pd_fs.participant_id.unique()[:int(np.floor(len(pd_fs.participant_id.unique())*.7))]
train_data_pd = pd_fs[pd_fs['participant_id'].isin(train_parts_pd)]
val_data_pd = pd_fs[~pd_fs['participant_id'].isin(train_parts_pd)]

train_parts_ad = ad_fs.participant_id.unique()[:int(np.floor(len(ad_fs.participant_id.unique())*.7))]
train_data_ad = ad_fs[ad_fs['participant_id'].isin(train_parts_ad)]
val_data_ad = ad_fs[~ad_fs['participant_id'].isin(train_parts_ad)]

#for each disease condition we sample the same amount of control participants
#then combine their data with the patient data to form our final training and validation sets
ct_train_parts_pd = ct_fs.participant_id.drop_duplicates().sample(len(train_parts_pd),random_state=rng).values
train_data_pd = pd.concat([train_data_pd,ct_fs[ct_fs.participant_id.isin(ct_train_parts_pd)]])

ct_train_parts_ad = ct_fs.participant_id.drop_duplicates().sample(len(train_parts_ad),random_state=rng).values
train_data_ad = pd.concat([train_data_ad,ct_fs[ct_fs.participant_id.isin(ct_train_parts_ad)]])

ct_val_parts_pd = ct_fs[~ct_fs['participant_id'].isin(ct_train_parts_pd)].participant_id.drop_duplicates().sample(len(val_data_pd.participant_id.unique()),random_state=rng).values
val_data_pd = pd.concat([val_data_pd,ct_fs[ct_fs.participant_id.isin(ct_val_parts_pd)]])

ct_val_parts_ad = ct_fs[~ct_fs['participant_id'].isin(ct_train_parts_ad)].participant_id.drop_duplicates().sample(len(val_data_ad.participant_id.unique()),random_state=rng).values
val_data_ad = pd.concat([val_data_ad,ct_fs[ct_fs.participant_id.isin(ct_val_parts_ad)]])

In [65]:
#how much data do we have in each condition?
print(f"""
PD: {len(train_data_pd)} training samples & {len(val_data_pd)} validation samples 
AD: {len(train_data_ad)} training samples & {len(val_data_ad)} validation samples
""")


PD: 147 training samples & 75 validation samples 
AD: 189 training samples & 93 validation samples



## ML benchmarking

Logistic regression, gradient boosting, support-vector machines, and neural networks are some common approaches to classification. PCA joins L1-penalized regression and ensemble selection methods as some common approaches to reducing the dimensionality of a predictor space to benefit the model of a classification problem. For these methods there also exists a considerable space of hyperparameters whose selection has consequence for the quality of the resulting model. We can't fully account for all of the possibilities in this work for practical reasons; rather, here we hope to survey some of the more-reasonable options.



In [49]:
pd_fs

,participant_id,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,...,local_shimmer,localDB_shimmer,apq3_shimmer,apq5_shimmer,apq11_shimmer,dda_shimmer,stoi,pesq,si_sdr,label
549,0581355f,23.116991,0.092754,21.342697,22.825933,24.566946,3.224249,225.074249,228.880539,32.518192,...,0.104231,1.006136,0.040367,0.057571,0.113731,0.121101,0.997695,3.452350,20.557886,1
550,0581355f,24.336529,0.113922,22.625807,23.774651,25.746820,3.121014,220.400818,196.087860,23.690027,...,0.103753,0.995240,0.042646,0.056899,0.101004,0.127937,0.994611,3.430656,20.712460,1
551,0581355f,27.139984,0.108037,24.597788,26.669912,29.847578,5.249790,150.989639,201.337662,29.131905,...,0.089274,0.879399,0.034279,0.046043,0.087223,0.102836,0.998030,3.640618,20.371683,1
970,0a282597,21.836462,0.041256,21.116489,21.717064,22.514534,1.398045,24.020990,10.231981,11.342595,...,0.099733,1.041104,0.031268,0.057010,0.128450,0.093804,0.983950,3.196664,19.711241,1
971,0a282597,22.335005,0.069886,21.427071,22.440836,23.452116,2.025045,175.395859,278.195953,17.169792,...,0.078371,0.783085,0.023630,0.038772,0.075018,0.070889,0.992179,3.646724,19.380545,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16857,df1453e7,25.831028,0.093940,23.968250,26.149540,27.952312,3.984062,107.598289,143.687622,35.949532,...,0.083193,0.855394,0.026366,0.042155,0.088318,0.079099,0.996015,3.829423,26.214901,1
16858,df1453e7,26.975775,0.112142,24.379810,26.953659,29.841303,5.461493,138.949738,175.578339,35.503948,...,0.097640,0.961507,0.035512,0.053817,0.103054,0.106535,0.998636,3.799230,23.188353,1
17997,eac8afdc,29.653240,0.094935,28.240120,29.455509,31.657101,3.416981,208.218246,212.480804,19.455433,...,0.111467,1.101201,0.046202,0.065270,0.111243,0.138607,0.985986,2.655652,19.009077,1
17998,eac8afdc,30.443380,0.067866,29.129333,30.194418,32.041504,2.912170,69.210289,106.138969,20.842350,...,0.119379,1.164370,0.052575,0.072507,0.120659,0.157726,0.965903,2.398106,17.836874,1


In [8]:
pd_stats[pd_stats['task_name'].isin(['Free-Speech','Free-speech-1'])]

,participant_id,session_id,task_name,transcription,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,...,ddp_jitter,local_shimmer,localDB_shimmer,apq3_shimmer,apq5_shimmer,apq11_shimmer,dda_shimmer,stoi,pesq,si_sdr
549,0581355f,21C1F1CA,Free-speech-1,NaN,23.116991,0.092754,21.342697,22.825933,24.566946,3.224249,...,0.030459,0.104231,1.006136,0.040367,0.057571,0.113731,0.121101,0.997695,3.452350,20.557886
970,0a282597,A6036FFA,Free-speech-1,To help patients get died died Diagnosis in th...,21.836462,0.041256,21.116489,21.717064,22.514534,1.398045,...,0.030252,0.099733,1.041104,0.031268,0.057010,0.128450,0.093804,0.983950,3.196664,19.711241
1254,0e3bbb6d,2CD1E0D6,Free-speech-1,All right. I'm interested in this study. My do...,31.424461,0.111820,31.076942,31.901854,32.999298,1.922356,...,0.034186,0.096911,0.954977,0.039595,0.049407,0.093868,0.118785,0.995881,3.324238,12.470948
1369,133372ef,BD54C6AC,Free-speech-1,NaN,26.253649,0.106699,24.143305,26.967258,28.319017,4.175713,...,0.035321,0.090628,0.940238,0.027054,0.044905,0.107312,0.081161,0.996596,3.519722,23.528757
1455,15443c3f,2F4F778F,Free-speech-1,I am interested in this study because it may h...,26.334810,0.243612,20.693600,27.649258,31.581661,10.888062,...,0.076247,0.162626,1.409223,0.075435,0.094270,0.131450,0.226305,0.948868,3.590131,24.354523
1639,16cce4e8,E88944CE,Free-speech-1,I am interested in this study because I have P...,33.155609,0.094500,32.414997,33.386608,34.512730,2.097733,...,0.028607,0.062906,0.705381,0.019396,0.028500,0.055283,0.058188,0.955044,2.620143,19.079657
1778,17578482,A8D26790,Free-speech-1,I'm interested in this study because I have Pa...,28.404694,0.096934,26.821936,28.278086,29.491602,2.669666,...,0.017054,0.099899,0.926064,0.043438,0.055102,0.087884,0.130315,0.924505,1.707437,19.631948
3126,27c3e7b2,49A6BFDB,Free-speech-1,NaN,25.442055,0.304191,13.605282,29.356421,30.902922,17.297640,...,0.022730,0.078382,0.805024,0.033299,0.039063,0.064356,0.099896,0.933973,1.888347,9.216987
3273,282b529e,84731907,Free-speech-1,I'm interested in this study for several reaso...,27.778841,0.068489,26.433769,27.679230,29.145910,2.712141,...,0.026038,0.080149,0.858649,0.022614,0.038007,0.080848,0.067841,0.990236,3.450804,25.828716
3519,2af5afbc,E7F3C75F,Free-speech-1,NaN,29.823526,0.069734,28.206337,29.622402,31.262657,3.056320,...,0.041592,0.124949,1.095971,0.057927,0.065555,0.097676,0.173780,0.998119,3.381133,17.890783
